# model ranking

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import os

SCI_STYLE = {
    'font.family': 'Arial',
    'font.size': 8,
    'axes.titlesize': 9,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'figure.dpi': 900,
    'savefig.format': 'tiff',
    'axes.linewidth': 0.8,
    'grid.linewidth': 0.5,
}

plt.rcParams.update(SCI_STYLE)

df = pd.read_csv('../final_report1.csv')

custom_model_order = ['JointSyn', 'CGMS', 'DeepDDS', 'MTLSynergy', 'MLP', 'XGB', 'RF']

target_mapping = {
    'synergy_loewe': 'Loewe',
    'synergy_hsa': 'HSA',
    'synergy_bliss': 'Bliss',
    'synergy_zip': 'ZIP'
}

metrics_config = {
    'MSE': {
        'metric_col': 'MSE_Mean',
        'title_suffix': 'MSE (Lower is better)',
        'ranking_direction': 'ascending'
    },
    'PCC': {
        'metric_col': 'PCC_Mean',
        'title_suffix': 'PCC (Higher is better)',
        'ranking_direction': 'descending'
    },
    'R2': {
        'metric_col': 'R2_Mean',
        'title_suffix': 'R-squared (Higher is better)',
        'ranking_direction': 'descending'
    },
    'RMSE': {
        'metric_col': 'RMSE_Mean',
        'title_suffix': 'RMSE (Lower is better)',
        'ranking_direction': 'ascending'
    },
    'MAE': {
        'metric_col': 'MAE_Mean',
        'title_suffix': 'MAE (Lower is better)',
        'ranking_direction': 'ascending'
    }
}

rank_colors = {
    1: '#8da0cb',  
    2: '#fc8d62',  
    3: '#66c2a5',  
}

targets = sorted(df['Target'].unique())

for metric, config in metrics_config.items():
    metric_df = df.copy()
    
    def rank_within_study_target(group):
        metric_col = config['metric_col']
        if config['ranking_direction'] == 'ascending':
            group['rank'] = group[metric_col].rank(method='min', ascending=True)
        else:
            group['rank'] = group[metric_col].rank(method='min', ascending=False)
        return group

    ranked_df = metric_df.groupby(['Study', 'Target']).apply(rank_within_study_target).reset_index(drop=True)

    ranked_df['rank'] = ranked_df['rank'].astype(int)
    top3_df = ranked_df[ranked_df['rank'] <= 3]

    fig, axs = plt.subplots(2, 2, figsize=(3.5, 4.5), 
                           sharex=True, sharey=True,
                           gridspec_kw={'wspace': 0.05, 'hspace': 0.15})

    for i, target in enumerate(targets):
        row_idx = i // 2
        col_idx = i % 2
        ax = axs[row_idx, col_idx]

        target_top3 = top3_df[top3_df['Target'] == target]

        rank_counts = target_top3.groupby(['Model', 'rank']).size().unstack(fill_value=0)

        for model in custom_model_order:
            if model not in rank_counts.index:
                rank_counts.loc[model] = [0, 0, 0]

        rank_counts = rank_counts.reindex(columns=[1, 2, 3], fill_value=0)

        rank1 = rank_counts.get(1, pd.Series(0, index=rank_counts.index))
        rank2 = rank_counts.get(2, pd.Series(0, index=rank_counts.index))
        rank3 = rank_counts.get(3, pd.Series(0, index=rank_counts.index))

        rank1 = rank1.reindex(custom_model_order, fill_value=0)
        rank2 = rank2.reindex(custom_model_order, fill_value=0)
        rank3 = rank3.reindex(custom_model_order, fill_value=0)

        total_wins = rank1 + rank2 + rank3

        bar1 = ax.bar(
            np.arange(len(custom_model_order)), 
            rank1,
            label='Rank 1',
            color=rank_colors[1],
            edgecolor='none',
            width=0.8
        )
        
        bar2 = ax.bar(
            np.arange(len(custom_model_order)), 
            rank2,
            bottom=rank1,
            label='Rank 2',
            color=rank_colors[2],
            edgecolor='none',
            width=0.8
        )
        
        bar3 = ax.bar(
            np.arange(len(custom_model_order)), 
            rank3,
            bottom=rank1 + rank2,
            label='Rank 3',
            color=rank_colors[3],
            edgecolor='none',
            width=0.8
        )
        
        for j, rect in enumerate(bar1):
            height = rect.get_height()
            if height > 0:
                ax.text(
                    rect.get_x() + rect.get_width() / 2, 
                    height / 2, 
                    str(int(height)),
                    ha='center', va='center',
                    color='white', fontweight='bold', fontsize=7
                )
        
        for j, rect in enumerate(bar2):
            bottom = rect.get_y()
            height = rect.get_height()
            if height > 0:
                ax.text(
                    rect.get_x() + rect.get_width() / 2, 
                    bottom + height / 2, 
                    str(int(height)),
                    ha='center', va='center',
                    color='white', fontweight='bold', fontsize=7
                )
        
        for j, rect in enumerate(bar3):
            bottom = rect.get_y()
            height = rect.get_height()
            if height > 0:
                ax.text(
                    rect.get_x() + rect.get_width() / 2, 
                    bottom + height / 2, 
                    str(int(height)),
                    ha='center', va='center',
                    color='black' if rank_colors[3] == '#ffc07d' else 'white', 
                    fontweight='bold', fontsize=7
                )

        mapped_target = target_mapping.get(target, target)
        ax.set_title(f'{mapped_target}', fontsize=8, pad=6)

        if row_idx == 1:  
            ax.set_xticks(np.arange(len(custom_model_order)))
            ax.set_xticklabels(custom_model_order, rotation=45, ha='right', fontsize=7)
        else:
            ax.set_xticks([])

        max_total = total_wins.max() if not total_wins.empty else 3
        ax.set_ylim(0, max_total * 1.25)

        ax.grid(False) 

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_visible(True)
        ax.spines['left'].set_visible(True)

        ax.spines['bottom'].set_linewidth(0.8)
        ax.spines['left'].set_linewidth(0.8)

    fig.text(0.02, 0.5, 'Number of Times Ranked', va='center', rotation='vertical', fontsize=8)

    legend_patches = [
        mpatches.Patch(color=rank_colors[1], label='Rank 1'),
        mpatches.Patch(color=rank_colors[2], label='Rank 2'),
        mpatches.Patch(color=rank_colors[3], label='Rank 3'),
    ]

    fig.legend(
        handles=legend_patches, 
        loc='upper right', 
        bbox_to_anchor=(0.92, 0.91), 
        ncol=1,  
        fontsize=7,
        frameon=False
    )

    plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5)

    plt.savefig(f'top3_ranking_model_{metric}.tiff', dpi=900, bbox_inches='tight', facecolor='white')
    plt.close(fig)


C:\Users\84991\AppData\Local\Temp\ipykernel_10592\1748902903.py:254: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5)
C:\Users\84991\AppData\Local\Temp\ipykernel_10592\1748902903.py:254: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5)
C:\Users\84991\AppData\Local\Temp\ipykernel_10592\1748902903.py:254: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5)
C:\Users\84991\AppData\Local\Temp\ipykernel_10592\1748902903.py:254: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95], pad=0.5)
C:\Users\84991\AppData\Local\Temp\ip

所有指标的Top 3模型排名分布图已按SCI要求生成
